In [3]:

# ============================================================
# PHASE 8: FORECASTING FRAMEWORK AND VALIDATION DESIGN
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project directory detection
# ------------------------------------------------------------

cwd = os.getcwd()

if os.path.basename(cwd).lower() == "notebooks":
    PROJECT_ROOT = os.path.dirname(cwd)
else:
    PROJECT_ROOT = cwd

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project directory:", PROJECT_ROOT)

# ------------------------------------------------------------
# Automatically locate the processed dataset
# ------------------------------------------------------------

search_directories = [
    PROCESSED_DIR,
    os.path.join(DATA_DIR, "interim"),
    os.path.join(DATA_DIR, "raw")
]

csv_files = []

for directory in search_directories:
    if os.path.exists(directory):
        for root, dirs, files in os.walk(directory):
            for filename in files:
                if filename.lower().endswith(".csv"):
                    csv_files.append(
                        os.path.join(root, filename)
                    )

if not csv_files:
    raise FileNotFoundError(
        "No CSV dataset found inside data/raw, data/interim "
        "or data/processed."
    )

print("\nAvailable CSV files:")

for i, file in enumerate(csv_files):
    print(i, os.path.relpath(file, PROJECT_ROOT))

# Prefer feature-engineered and processed files.
priority_keywords = [
    "feature_engineered",
    "feature",
    "processed",
    "cleaned",
    "transformed"
]

def file_priority(filepath):
    name = os.path.basename(filepath).lower()

    for index, keyword in enumerate(priority_keywords):
        if keyword in name:
            return index

    return len(priority_keywords)

csv_files.sort(key=file_priority)

# Load candidate datasets and identify a usable one.
date_candidates = [
    "date", "report_date", "observation_date",
    "snapshot_date", "month", "year"
]

target_candidates = [
    "total_children",
    "total_care_load",
    "care_load",
    "placement_demand",
    "total",
    "count",
    "value"
]

df = None
date_col = None
target_col = None
selected_file = None

for filepath in csv_files:

    try:
        candidate = pd.read_csv(filepath)

        candidate.columns = (
            candidate.columns.astype(str).str.strip()
        )

        date_match = next(
            (
                c for c in date_candidates
                if c.lower() in [
                    col.lower() for col in candidate.columns
                ]
            ),
            None
        )

        if date_match is None:
            continue

        actual_date_col = next(
            c for c in candidate.columns
            if c.lower() == date_match.lower()
        )

        candidate[actual_date_col] = pd.to_datetime(
            candidate[actual_date_col],
            errors="coerce"
        )

        numeric_columns = candidate.select_dtypes(
            include=np.number
        ).columns.tolist()

        target_match = next(
            (
                c for c in target_candidates
                if c.lower() in [
                    col.lower() for col in candidate.columns
                ]
            ),
            None
        )

        if target_match is not None:
            actual_target_col = next(
                c for c in candidate.columns
                if c.lower() == target_match.lower()
            )
        else:
            excluded = [
                c for c in numeric_columns
                if any(
                    word in c.lower()
                    for word in [
                        "lag", "rolling", "forecast",
                        "year", "month", "day", "index"
                    ]
                )
            ]

            available_targets = [
                c for c in numeric_columns
                if c not in excluded
            ]

            if not available_targets:
                continue

            actual_target_col = available_targets[0]

        valid_rows = (
            candidate[actual_date_col].notna()
            & pd.to_numeric(
                candidate[actual_target_col],
                errors="coerce"
            ).notna()
        )

        if valid_rows.sum() < 20:
            continue

        df = candidate.loc[valid_rows].copy()
        date_col = actual_date_col
        target_col = actual_target_col
        selected_file = filepath

        break

    except Exception as error:
        print("Skipping:", os.path.basename(filepath), error)

if df is None:
    raise ValueError(
        "No suitable dataset found. "
        "A CSV with at least 20 valid dated numeric observations "
        "is required."
    )

print("\nSelected dataset:", selected_file)
print("Date column:", date_col)
print("Target column:", target_col)

df[target_col] = pd.to_numeric(
    df[target_col], errors="coerce"
)

df = (
    df.dropna(subset=[date_col, target_col])
    .sort_values(date_col)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Define forecasting target and forecast origin
# ------------------------------------------------------------

df["forecast_origin"] = df[date_col]

# ------------------------------------------------------------
# Determine reporting frequency
# ------------------------------------------------------------

date_differences = df[date_col].diff().dropna()

median_interval = date_differences.median()

if median_interval <= pd.Timedelta(days=1.5):
    frequency = "Daily"
elif median_interval <= pd.Timedelta(days=8):
    frequency = "Weekly"
else:
    frequency = "Monthly_or_Lower_Frequency"

print("\nReporting frequency:", frequency)
print("Median interval:", median_interval)

# ------------------------------------------------------------
# Define forecast horizons
# ------------------------------------------------------------

horizon_days = [1, 7, 14, 30]

if frequency == "Daily":
    horizon_steps = {
        "1_day": 1,
        "7_day": 7,
        "14_day": 14,
        "30_day": 30
    }
elif frequency == "Weekly":
    horizon_steps = {
        "1_day": 1,
        "7_day": 1,
        "14_day": 2,
        "30_day": 5
    }
else:
    horizon_steps = {
        "1_day": 1,
        "7_day": 1,
        "14_day": 1,
        "30_day": 1
    }

# ------------------------------------------------------------
# Create forecast targets and actual target dates
# ------------------------------------------------------------

for horizon_name, steps in horizon_steps.items():

    df[f"target_{horizon_name}"] = (
        df[target_col].shift(-steps)
    )

    df[f"actual_date_{horizon_name}"] = (
        df[date_col].shift(-steps)
    )

max_steps = max(horizon_steps.values())

forecast_df = df.iloc[:-max_steps].copy()

forecast_df.reset_index(drop=True, inplace=True)

n = len(forecast_df)

if n < 20:
    raise ValueError(
        "Insufficient observations after reserving forecast horizons."
    )

# ------------------------------------------------------------
# Chronological train-validation-test split
# ------------------------------------------------------------

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_data = forecast_df.iloc[:train_end].copy()

validation_data = forecast_df.iloc[
    train_end:validation_end
].copy()

test_data = forecast_df.iloc[
    validation_end:
].copy()

print("\nCHRONOLOGICAL DATA SPLIT")

print("Total observations:", n)
print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Testing:", len(test_data))

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Testing"],
    "Observations": [
        len(train_data),
        len(validation_data),
        len(test_data)
    ],
    "Start_Date": [
        train_data[date_col].min(),
        validation_data[date_col].min(),
        test_data[date_col].min()
    ],
    "End_Date": [
        train_data[date_col].max(),
        validation_data[date_col].max(),
        test_data[date_col].max()
    ]
})

print("\n", split_summary.to_string(index=False))

# ------------------------------------------------------------
# Chronological train-test separation
# ------------------------------------------------------------

X_train = train_data[[target_col]].copy()
X_validation = validation_data[[target_col]].copy()
X_test = test_data[[target_col]].copy()

y_train = train_data["target_30_day"].copy()
y_validation = validation_data["target_30_day"].copy()
y_test = test_data["target_30_day"].copy()

# ------------------------------------------------------------
# Training-only transformation
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

print("\nTraining-only transformation completed.")

# ------------------------------------------------------------
# Expanding-window validation
# ------------------------------------------------------------

validation_pool = forecast_df.iloc[:validation_end]

number_of_splits = min(
    5,
    len(validation_pool) // 5
)

if number_of_splits < 2:
    raise ValueError(
        "Insufficient observations for expanding-window validation."
    )

expanding_cv = TimeSeriesSplit(
    n_splits=number_of_splits
)

expanding_results = []

for fold, (train_idx, val_idx) in enumerate(
    expanding_cv.split(validation_pool), start=1
):

    fold_train = validation_pool.iloc[train_idx]
    fold_validation = validation_pool.iloc[val_idx]

    expanding_results.append({
        "Fold": fold,
        "Training_Start": fold_train[date_col].min(),
        "Training_End": fold_train[date_col].max(),
        "Validation_Start": fold_validation[date_col].min(),
        "Validation_End": fold_validation[date_col].max(),
        "Training_Size": len(fold_train),
        "Validation_Size": len(fold_validation)
    })

expanding_results = pd.DataFrame(expanding_results)

# ------------------------------------------------------------
# Rolling-window validation
# ------------------------------------------------------------

rolling_results = []

window_size = max(5, int(train_end * 0.70))

step_size = max(1, len(validation_data) // 5)

for start in range(
    0,
    len(validation_pool) - window_size,
    step_size
):

    rolling_train = validation_pool.iloc[
        start:start + window_size
    ]

    rolling_validation = validation_pool.iloc[
        start + window_size:
        start + window_size + step_size
    ]

    if rolling_validation.empty:
        continue

    rolling_results.append({
        "Training_Start": rolling_train[date_col].min(),
        "Training_End": rolling_train[date_col].max(),
        "Validation_Start": rolling_validation[date_col].min(),
        "Validation_End": rolling_validation[date_col].max(),
        "Training_Size": len(rolling_train),
        "Validation_Size": len(rolling_validation)
    })

rolling_results = pd.DataFrame(rolling_results)

# ------------------------------------------------------------
# Walk-forward evaluation
# ------------------------------------------------------------

walk_forward_results = []

for horizon_name in horizon_steps:

    for _, row in validation_data.iterrows():

        actual = row[f"target_{horizon_name}"]

        if pd.isna(actual):
            continue

        predicted = row[target_col]

        walk_forward_results.append({
            "Forecast_Origin": row["forecast_origin"],
            "Actual_Target_Date": row[
                f"actual_date_{horizon_name}"
            ],
            "Horizon": horizon_name,
            "Actual": actual,
            "Predicted": predicted,
            "Absolute_Error": abs(actual - predicted)
        })

walk_forward_predictions = pd.DataFrame(
    walk_forward_results
)

# ------------------------------------------------------------
# Leakage prevention checks
# ------------------------------------------------------------

assert (
    train_data[date_col].max()
    < validation_data[date_col].min()
)

assert (
    validation_data[date_col].max()
    < test_data[date_col].min()
)

assert (
    walk_forward_predictions["Actual_Target_Date"]
    > walk_forward_predictions["Forecast_Origin"]
).all()

# ------------------------------------------------------------
# Save validation predictions
# ------------------------------------------------------------

validation_predictions = walk_forward_predictions.copy()

# ------------------------------------------------------------
# Save forecasting framework outputs
# ------------------------------------------------------------

forecast_df.to_csv(
    os.path.join(OUTPUT_DIR, "forecasting_framework_dataset.csv"),
    index=False
)

split_summary.to_csv(
    os.path.join(OUTPUT_DIR, "chronological_split_summary.csv"),
    index=False
)

expanding_results.to_csv(
    os.path.join(OUTPUT_DIR, "expanding_window_validation.csv"),
    index=False
)

rolling_results.to_csv(
    os.path.join(OUTPUT_DIR, "rolling_window_validation.csv"),
    index=False
)

walk_forward_predictions.to_csv(
    os.path.join(OUTPUT_DIR, "walk_forward_predictions.csv"),
    index=False
)

validation_predictions.to_csv(
    os.path.join(OUTPUT_DIR, "validation_predictions.csv"),
    index=False
)

# ------------------------------------------------------------
# Save forecasting configuration
# ------------------------------------------------------------

configuration = {
    "source_file": os.path.relpath(selected_file, PROJECT_ROOT),
    "target_column": target_col,
    "date_column": date_col,
    "frequency": frequency,
    "median_reporting_interval": str(median_interval),
    "forecast_horizons_days": horizon_days,
    "forecast_horizon_steps": horizon_steps,
    "total_observations": len(df),
    "usable_observations": n,
    "training_observations": len(train_data),
    "validation_observations": len(validation_data),
    "testing_observations": len(test_data),
    "split_design": "Chronological 70/15/15 initial design",
    "validation_methods": [
        "Expanding-window",
        "Rolling-window",
        "Walk-forward"
    ],
    "transformation_policy": "Training data only",
    "overlapping_horizons": "Separate forecast origins"
}

with open(
    os.path.join(OUTPUT_DIR, "forecasting_configuration.json"),
    "w"
) as file:

    json.dump(configuration, file, indent=4, default=str)

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\n============================================")
print("PHASE 8 EXECUTION COMPLETED")
print("============================================")

print("Selected file:", os.path.basename(selected_file))
print("Target:", target_col)
print("Frequency:", frequency)
print("Forecast horizons:", horizon_days)

print("\nExpanding-window folds:", len(expanding_results))
print("Rolling-window folds:", len(rolling_results))
print("Walk-forward predictions:", len(walk_forward_predictions))
print("Validation predictions:", len(validation_predictions))

print("\nAll output files saved in:", OUTPUT_DIR)
print("Chronological leakage checks passed.")


Project directory: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand

Available CSV files:
0 data\processed\cleaned_care_data.csv
1 data\processed\data_dictionary.csv
2 data\processed\engineered_features.csv
3 data\processed\feature_availability_report.csv
4 data\processed\feature_correlation.csv
5 data\processed\feature_dictionary.csv
6 data\processed\feature_leakage_report.csv
7 data\processed\high_multicollinearity_pairs.csv
8 data\processed\initial_data_profile.csv
9 data\raw\HHS_Unaccompanied_Alien_Children_Program.csv

Selected dataset: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\data\processed\engineered_features.csv
Date column: date
Target column: children_in_hhs_care

Reporting frequency: Daily
Median interval: 1 days 00:00:00

CHRONOLOGICAL DATA SPLIT
Total observations: 690
Training: 482
Validation: 104
Testing: 104

    Dataset  Observations Start_Date   End_Date
  Training           482 2023-01-12 2025-01-01
Validation           104